# 04 - Pose Inference
Run the trained pose model on all videos to extract keypoint coordinates.

In [1]:
# ===== CONFIGURATION =====
# GitHub -- do not change
GITHUB_REPO_URL = "https://github.com/kaarthik-balakrishnan/LightningPoseTrack.git"
GIT_BRANCH = "main"

# Google Drive root -- change this to match your Drive structure
DRIVE_ROOT = "/content/drive/My Drive/PigBehavior"  # <-- SET THIS to your root

# Derived paths (change if your folders are at custom locations)
DRIVE_RAW_VIDEOS = f"{DRIVE_ROOT}/raw_videos"               # Input: session video folders
DRIVE_MODELS = f"{DRIVE_ROOT}/trained_models"         # Input: trained model checkpoints
DRIVE_POSE_OUTPUTS = f"{DRIVE_ROOT}/pose_outputs"     # Output: pose prediction CSVs

CONFIDENCE_THRESHOLD = 0.5

# Per-camera pixel bounds (corners of the visible arena in order:
# top-left, top-right, bottom-right, bottom-left).
# Any keypoint predicted outside these bounds is discarded.
PIXEL_POINTS = {
    1: [(153, 388), (1415, 388), (1415, 907), (153, 907)],
    2: [(252, 464), (1754, 432), (1761, 640), (249, 640)],
    3: [(166, 421), (1537, 453), (1535, 610), (160, 608)],
    4: [(360, 360), (1348, 384), (1368, 840), (367, 835)],
}

# Google Drive folder ID (for reference)
DRIVE_FOLDER_ID = "1X_41ZW3HfwVeft2lPld3XNqXsdxRDIwb"

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os, sys
REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

Cloning into '/content/LightningPoseTrack'...
remote: Enumerating objects: 1400, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 1400 (delta 109), reused 69 (delta 31), pack-reused 1237 (from 1)
Receiving objects: 100% (1400/1400), 345.49 MiB | 34.82 MiB/s, done.
Resolving deltas: 100% (283/283), done.
/content/LightningPoseTrack


In [4]:
# Install Lightning Pose + inference dependencies
!pip install --quiet "lightning-pose[all]" opencv-python pandas numpy pyarrow imageio[ffmpeg]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 827.9/827.9 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.9/129.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.8/155.8 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 964.3/964.3 kB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.3/220.3 kB 22.3 MB/

In [5]:
from pathlib import Path

model_dir = Path(DRIVE_MODELS) / "pose_model"
if model_dir.exists() and (model_dir / "config.yaml").exists():
    print(f"Model directory found: {model_dir}")
    print(f"  Contents: {[p.name for p in model_dir.iterdir()]}")
else:
    print(f"Model directory not found at {model_dir}. Complete notebook 03 first.")

Model directory found: /content/drive/My Drive/PigBehavior/trained_models/pose_model
  Contents: ['tb_logs', 'config.yaml', 'CollectedData_LP.csv', 'image_preds', 'predictions_pixel_error.csv', 'train_status.json', 'predictions.csv', 'predictions_pca_singleview_error.csv']


In [6]:
from src.io.video_inventory import scan_videos, parse_camera_from_filename

df = scan_videos(DRIVE_RAW_VIDEOS)
print(f"Found {len(df)} videos to process")
df[["filename", "session", "camera", "frame_count", "duration_min"]]

  Root path: /content/drive/My Drive/PigBehavior/raw_videos
  Path exists: True
  First 20 entries: ['20260825_Behaving']
  133738-4.ASF — OK (h264, 1920x1080, 10.0 fps, 112f)
  133738-4.mp4 — OK (h264, 1920x1080, 10.0 fps, 42f)
  133740-3.ASF — OK (h264, 1920x1080, 10.0 fps, 3001f)
  133740-3.mp4 — OK (h264, 1920x1080, 10.0 fps, 2854f)
  133743-1.ASF — OK (h264, 1920x1080, 10.2 fps, 369f)
  133743-1.mp4 — OK (h264, 1920x1080, 10.2 fps, 147f)
  133743-2.ASF — OK (h264, 1920x1080, 10.2 fps, 3094f)
  133743-2.mp4 — OK (h264, 1920x1080, 10.2 fps, 2184f)
  133752-4.ASF — OK (h264, 1920x1080, 5.0 fps, 1501f)
  133752-4.mp4 — OK (h264, 1920x1080, 5.0 fps, 1425f)
  133820-1.ASF — OK (h264, 1920x1080, 10.2 fps, 420f)
  133820-1.mp4 — OK (h264, 1920x1080, 10.2 fps, 125f)
  133916-1.ASF — OK (h264, 1920x1080, 10.2 fps, 3076f)
  133916-1.mp4 — OK (h264, 1920x1080, 10.2 fps, 1627f)
  134239-3.ASF — OK (h264, 1920x1080, 10.0 fps, 1312f)
  134239-3.mp4 — OK (h264, 1920x1080, 10.0 fps, 1260f)
  13425

,filename,session,camera,frame_count,duration_min
0,133738-4.ASF,20260825_Behaving,4,112,0.19
1,133738-4.mp4,20260825_Behaving,4,42,0.07
2,133740-3.ASF,20260825_Behaving,3,3001,5.00
3,133740-3.mp4,20260825_Behaving,3,2854,4.76
4,133743-1.ASF,20260825_Behaving,1,369,0.60
5,133743-1.mp4,20260825_Behaving,1,147,0.24
6,133743-2.ASF,20260825_Behaving,2,3094,5.03
7,133743-2.mp4,20260825_Behaving,2,2184,3.55
8,133752-4.ASF,20260825_Behaving,4,1501,5.00
9,133752-4.mp4,20260825_Behaving,4,1425,4.75


In [7]:
import cv2
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

from lightning_pose.api import Model

# Precompute per-camera bounding boxes from PIXEL_POINTS
CAMERA_BOUNDS = {}
for cam, corners in PIXEL_POINTS.items():
    xs = [p[0] for p in corners]
    ys = [p[1] for p in corners]
    CAMERA_BOUNDS[cam] = (min(xs), max(xs), min(ys), max(ys))
print("Camera bounds (x_min, x_max, y_min, y_max):")
for cam, (xmn, xmx, ymn, ymx) in sorted(CAMERA_BOUNDS.items()):
    print(f"  Camera {cam}: x=[{xmn}, {xmx}]  y=[{ymn}, {ymx}]")

def filter_out_of_bounds(kps, confs, cam):
    """Set keypoints outside the camera's pixel bounds to NaN."""
    if cam not in CAMERA_BOUNDS:
        return kps, confs
    xmin, xmax, ymin, ymax = CAMERA_BOUNDS[cam]
    n_kp = len(kps) // 2
    for i in range(n_kp):
        x, y = kps[i*2], kps[i*2+1]
        if x < xmin or x > xmax or y < ymin or y > ymax:
            kps[i*2] = float("nan")
            kps[i*2+1] = float("nan")
            confs[i] = 0.0
    return kps, confs

# Load the trained model from the model directory (contains config.yaml + .ckpt)
model_dir = Path(DRIVE_MODELS) / "pose_model"
model = Model.from_dir(str(model_dir))
print(f"Model loaded from {model_dir}")

pose_output_dir = Path(DRIVE_POSE_OUTPUTS)
pose_output_dir.mkdir(parents=True, exist_ok=True)

KEYPOINT_NAMES = [
    "snout", "left_ear", "right_ear", "neck",
    "shoulders", "mid_back", "hip", "tail_base",
]

for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing videos"):
    video_path = Path(DRIVE_RAW_VIDEOS) / row["path"]
    session = row["session"]
    camera = row["camera"]
    stem = Path(row["filename"]).stem

    out_file = pose_output_dir / session / f"{stem}_cam{camera}_pose.parquet"
    if out_file.exists():
        print(f"Skipping {out_file.name} (already exists)")
        continue

    # Use OpenCV to read frames (DALI cannot handle .asf)
    cap = cv2.VideoCapture(str(video_path))
    kp_list, conf_list = [], []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = model.predict_frame(frame_rgb)
        kps = result["keypoints"].ravel()
        confs = result["confidence"]
        kps, confs = filter_out_of_bounds(kps, confs, camera)
        kp_list.append(kps)
        conf_list.append(confs)
    cap.release()

    # Build DataFrame matching the same column schema as predict_on_video_file output
    n_kp = len(KEYPOINT_NAMES)
    gen_df = pd.DataFrame({
        "frame": range(len(kp_list)),
        **{f"{kp}_x": [k[i*2] for k in kp_list] for i, kp in enumerate(KEYPOINT_NAMES)},
        **{f"{kp}_y": [k[i*2+1] for k in kp_list] for i, kp in enumerate(KEYPOINT_NAMES)},
        **{f"{kp}_likelihood": [c[i] for c in conf_list] for i, kp in enumerate(KEYPOINT_NAMES)},
    })
    out_file.parent.mkdir(parents=True, exist_ok=True)
    gen_df.to_parquet(str(out_file))
    print(f"Saved: {out_file} ({len(gen_df)} frames)")

print("\nPose inference complete!")


Camera bounds (x_min, x_max, y_min, y_max):
  Camera 1: x=[153, 1415]  y=[388, 907]
  Camera 2: x=[249, 1761]  y=[432, 640]
  Camera 3: x=[160, 1537]  y=[421, 610]
  Camera 4: x=[360, 1368]  y=[360, 840]
Model loaded from /content/drive/My Drive/PigBehavior/trained_models/pose_model


Processing videos:   0%|          | 0/46 [00:00<?, ?it/s]

	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL omegaconf.dictconfig.DictConfig was not an allowed global by default. Please use `torch.serialization.add_safe_globals([omegaconf.dictconfig.DictConfig])` or the `torch.serialization.safe_globals([omegaconf.dictconfig.DictConfig])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth



  0%|          | 0.00/97.8M [00:00<?, ?B/s]
 12%|█▏        | 12.0M/97.8M [00:00<00:00, 125MB/s]
 25%|██▌       | 24.9M/97.8M [00:00<00:00, 131MB/s]
 39%|███▉      | 38.5M/97.8M [00:00<00:00, 136MB/s]
 60%|██████    | 59.0M/97.8M [00:00<00:00, 167MB/s]
 78%|███████▊  | 76.0M/97.8M [00:00<00:00, 170MB/s]
100%|██████████| 97.8M/97.8M [00:00<00:00, 161MB/s]


Saved: /content/drive/My Drive/PigBehavior/pose_outputs/20260825_Behaving/133738-4_cam4_pose.parquet (42 frames)
Skipping 133738-4_cam4_pose.parquet (already exists)
Saved: /content/drive/My Drive/PigBehavior/pose_outputs/20260825_Behaving/133740-3_cam3_pose.parquet (2854 frames)
Skipping 133740-3_cam3_pose.parquet (already exists)
Saved: /content/drive/My Drive/PigBehavior/pose_outputs/20260825_Behaving/133743-1_cam1_pose.parquet (147 frames)
Skipping 133743-1_cam1_pose.parquet (already exists)
Saved: /content/drive/My Drive/PigBehavior/pose_outputs/20260825_Behaving/133743-2_cam2_pose.parquet (2184 frames)
Skipping 133743-2_cam2_pose.parquet (already exists)
Saved: /content/drive/My Drive/PigBehavior/pose_outputs/20260825_Behaving/133752-4_cam4_pose.parquet (1516 frames)
Skipping 133752-4_cam4_pose.parquet (already exists)
Saved: /content/drive/My Drive/PigBehavior/pose_outputs/20260825_Behaving/133820-1_cam1_pose.parquet (125 frames)
Skipping 133820-1_cam1_pose.parquet (already exis

In [8]:
# Verify outputs
output_files = list(pose_output_dir.rglob("*.parquet"))
print(f"Total pose output files: {len(output_files)}")
for f in output_files:
    print(f"  {f.relative_to(pose_output_dir)}")

Total pose output files: 126
  260529.00000003/161308-1_cam1_pose.parquet
  260529.00000003/161311-2_cam2_pose.parquet
  260529.00000003/161312-3_cam3_pose.parquet
  260529.00000003/161312-4_cam4_pose.parquet
  260529.00000003/161437-2_cam2_pose.parquet
  260529.00000003/161454-3_cam3_pose.parquet
  260529.00000003/161459-4_cam4_pose.parquet
  260529.00000003/161550-3_cam3_pose.parquet
  260529.00000003/161559-1_cam1_pose.parquet
  260529.00000003/161700-4_cam4_pose.parquet
  260529.00000003/161838-3_cam3_pose.parquet
  260529.00000003/161844-2_cam2_pose.parquet
  260529.00000003/161855-1_cam1_pose.parquet
  260529.00000003/161952-2_cam2_pose.parquet
  260529.00000003/162009-3_cam3_pose.parquet
  260529.00000003/162021-4_cam4_pose.parquet
  260529.00000003/162241-3_cam3_pose.parquet
  260529.00000003/162258-2_cam2_pose.parquet
  260529.00000003/162304-1_cam1_pose.parquet
  260529.00000003/162603-2_cam2_pose.parquet
  260529.00000003/162610-3_cam3_pose.parquet
  260529.00000003/162631-4